# 📗 서비스 키 관리와 사용량 기록: 키 발급, 폐기, 토큰 사용량을 SQLite에 저장하기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행 위치:</strong> VS Code의 Remote - SSH로 접속한 RunPod 파드 하나에서 노트북, 터미널, 두 서버를 모두 실행합니다. <strong style="font-size:1em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 교안 02(과제 02를 했다면 과제 02까지)를 마친 VS Code 원격 창에서 이어서 진행합니다.<br><strong style="font-size:1em;color:#1f3a5f;">서버:</strong> 터미널 1의 vLLM 서버는 <strong style="font-size:1em;color:#1f3a5f;">그대로 이어 씁니다.</strong> 터미널 2에서 관리자 키를 만들고 FastAPI 서버만 이 교안의 앱으로 바꿔 다시 띄웁니다.<br><strong style="font-size:1em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장(교안 01과 같은 파드).</div>

#### **서비스 키 관리(API key management)**

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서비스 키를 발급하고, 요청마다 확인하고, 키별 사용량을 기록하고, 필요할 때 폐기하는 일을 서빙 API가 직접 맡는 것</strong>입니다.</p>
<p style="margin:6px 0;">키를 파일에 적어 두면 키를 더하거나 끊을 때마다 파일을 고치고 서버를 다시 켜야 합니다. 사용자가 늘면 발급과 폐기는 관리자 주소로 처리하고, 요금과 사용 한도의 근거가 되는 토큰 사용량은 키별로 남겨야 합니다. 예를 들어 노무 상담 앱과 전월세 상담 앱에 키를 따로 발급하면 앱마다 쓴 토큰을 따로 집계할 수 있고, 노무 상담 앱의 키가 유출되어도 서버를 끄지 않고 그 키만 끊을 수 있습니다.</p>
<p style="margin:6px 0;">고객사마다 키를 발급하고 쓴 만큼 청구하는 유료 모델 API, 사내 앱별 사용량을 나눠 보는 상담 API, 유출된 키를 바로 끊어야 하는 서비스가 이렇게 키를 관리합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/41_key_lifecycle.png" width="1050" alt="서비스 키를 발급, 사용, 기록, 폐기하는 네 단계와, 각 단계가 읽고 쓰는 SQLite의 api_keys, usage 테이블">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02는 서버 패키지의 JSON 키 파일에서 서비스 키를 읽었습니다. 이 교안은 같은 노무, 전월세 창구를 SQLite 키 저장소에 연결한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.managed_main</code>을 실행해, 관리자 주소로 키를 발급하고 폐기하며 요청마다 토큰 사용량을 기록합니다. 교안 02의 요청 한도와 오류 응답 양식은 그대로 씁니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/40_lesson03_goals.png" width="1050" alt="학습 목표 그림: 1 SQLite 키 저장소, 2 키 발급, 3 사용량 기록, 4 키 폐기">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day77-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. 서비스 키를 해시로 저장하는 SQLite 키 저장소 코드를 읽고, DB 파일에 키 원문이 없는 것을 확인하며 원문 대신 해시를 저장하는 이유를 설명합니다.
2. 관리자 키로 보호한 주소에서 서비스 키를 발급하고, 발급한 키로 상담 주소를 호출합니다.
3. 서버가 요청마다 키별로 남긴 토큰 사용량을 사용자와 관리자가 각각 조회하고, 두 합계가 같은 것을 확인합니다.
4. 키를 폐기하고, 폐기한 키의 요청이 401로 돌아오는 것을 확인합니다.

</div>

## 1. 실습 준비

### 1-1. 터미널 2에서 관리자 키 생성

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1의 vLLM 서버는 그대로 둡니다.</strong> 파드를 새로 만들었거나 서버가 꺼졌다면 교안 01 1절(실습 준비와 vLLM 서버 실행)의 순서대로 키 생성과 vLLM 실행부터 마칩니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 앞 노트북의 앱(과제 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.secure_main</code> 또는 교안 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code>)을 끄고, 아래 블록으로 이 교안 폴더로 이동합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API/교안_03_서비스키_관리와_사용량_기록"
pwd   # 끝이 교안_03_서비스키_관리와_사용량_기록이면 맞습니다
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">관리자 키(admin key)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키 발급, 키 목록, 폐기, 사용량 조회 같은 관리자 주소를 부를 때 쓰는 키입니다. 서비스 키와 다른 키이고 사용자에게 나눠 주지 않습니다.</li>
<li style="margin:3px 0;">vLLM 키와 같이 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>로 보관합니다. FastAPI 서버는 이 값과 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Admin-Key</code> 헤더를 비교하고, 노트북은 이 값을 읽어 헤더에 담습니다.</li>
</ul>
<p style="margin:6px 0;">이 교안부터 키가 셋입니다. 세 키는 보내는 쪽, 받는 곳, 확인하는 방법이 모두 다릅니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">키</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">보내는 쪽</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">받는 곳과 헤더</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">서버가 확인하는 방법</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI 서버</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 서버, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Authorization</code> 헤더</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 서버가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code> 값과 비교합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">관리자 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">관리자(이 노트북)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">관리자 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/admin/keys</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Admin-Key</code> 헤더</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI 서버가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 값과 비교합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서비스 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">svc-</code>로 시작)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">상담 앱(노무 상담 앱 등)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">상담 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/labor/chat</code> 등, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI 서버가 해시로 바꿔 DB에서 찾습니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">같은 터미널 2에서 vLLM 키 생성과 같은 방식으로 관리자 키를 만들어 저장합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 관리자 키를 생성해 원격 저장소의 .env에 ADMIN_API_KEY로 저장합니다(키 값은 화면에 보이지 않습니다)
ADMIN_API_KEY="$(uv run python -c 'import secrets; print("enkoa-admin-" + secrets.token_urlsafe(32))')" &&
export ADMIN_API_KEY &&
uv run python -c 'import os; from dotenv import set_key; set_key("../../.env", "ADMIN_API_KEY", os.environ["ADMIN_API_KEY"])' &&
grep -c '^ADMIN_API_KEY=' ../../.env   # 1이 나오면 저장된 것입니다
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01 1-2의 vLLM 키 생성 블록과 같은 명령입니다. 다른 점은 키 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>)과 키 앞에 붙는 글자(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-admin-</code>)입니다. 저장하는 곳은 vLLM 키와 같은 원격 저장소(실습자료) 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>이고, 터미널 2는 이 교안 폴더에 있으므로 두 단계 위 경로(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">../../.env</code>)로 가리킵니다. 마지막 줄이 1이면 저장된 것이고, 키 값은 화면에 보이지 않습니다.</p>
<p style="margin:6px 0;">이 블록을 다시 실행하면 새 키로 바뀝니다. 그때는 아래 1-2의 FastAPI 서버와 1-3의 노트북 준비 셀을 다시 실행합니다.</p>
</div>

### 1-2. 터미널 2에서 키 관리 앱 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code>에는 교안 02의 완성 코드와 이 교안의 파일 다섯 개가 들어 있습니다. 이번 서버 코드에는 채울 빈칸이 없어, 코드를 읽고 그대로 띄웁니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">폴더, 파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">계층과 하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">보는 절</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">repositories/key_store.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">새 계층(데이터 접근): SQLite 파일에 키 해시와 사용량을 저장하고 조회합니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2절, 4절, 5절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/db_security.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">의존성: 요청의 서비스 키를 DB에서 찾고 요청 한도를 지킵니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/admin_router.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">표현 계층: 관리자 주소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/admin/keys</code>)로 발급, 목록, 폐기, 사용량 조회를 합니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3절, 4절, 5절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/metered_router.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">표현 계층: 상담 주소에서 답을 돌려주고 토큰 사용량을 기록합니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">managed_main.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">앱 조립: lifespan에서 DB 연결을 열고 테이블을 만들며, 주소 묶음 셋을 앱에 붙입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1절</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">DB를 읽고 쓰는 SQL은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">repositories/</code> 계층에만 둡니다. 라우터와 의존성은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">issue_key</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">find_active_key</code> 같은 함수만 부르므로, 나중에 SQLite를 다른 DB로 바꿀 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">repositories/key_store.py</code>만 고칩니다.</p>
<p style="margin:6px 0;">같은 터미널 2에서 앱을 띄웁니다. uvicorn 명령은 교안 02와 같고 모듈 이름만 다릅니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn serving_app.managed_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
<p style="margin:6px 0;">서버가 시작할 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lifespan</code>이 DB 연결 하나를 열고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">init_db()</code>로 이 교안 폴더에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/service.db</code> 파일과 두 테이블을 만듭니다. 이 연결은 앱 전체가 함께 쓰다가 서버를 끌 때 닫습니다. 터미널 2에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RuntimeError: ADMIN_API_KEY 가 없습니다</code>가 나오면 1-1의 관리자 키 블록을 실행했는지 확인합니다.</p>
</div>

```python
# managed_main.py
"""교안 03 서빙 API(DB 키 관리, 사용량 기록). 실행: uv run uvicorn serving_app.managed_main:app --host 0.0.0.0 --port 8080 --reload

교안 02 와 다른 점: 키를 SQLite 에 해시로 저장하고, 관리자 주소로 발급과 폐기를 하며, 요청마다 토큰 사용량을 기록한다.
"""
from contextlib import asynccontextmanager

import aiosqlite
from fastapi import FastAPI

from .core.errors import register_error_handlers
from .repositories.key_store import SERVICE_DB_PATH, init_db
from .routers.admin_router import admin_router
from .routers.metered_router import metered_router
from .routers.system_router import system_router
from .services.backend import vllm_client


@asynccontextmanager
async def lifespan(app: FastAPI):
    # 서버가 시작할 때 DB 연결 하나를 열어 앱 전체가 함께 쓴다(교안 00 5-3 과 같은 방식)
    # aiosqlite 는 이 연결의 스레드 하나에서 요청을 차례로 실행하므로, 여러 요청이 같은 연결을 써도 엉키지 않는다
    SERVICE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)  # output 폴더가 없으면 만든다
    connection = await aiosqlite.connect(SERVICE_DB_PATH)
    connection.row_factory = aiosqlite.Row  # 조회한 행의 값을 row["user"] 처럼 열 이름으로 꺼낸다
    await init_db(connection)  # 테이블을 만든다(이미 있으면 그대로 둔다)
    app.state.db = connection
    try:
        yield  # 준비가 끝나면 FastAPI가 요청을 처리한다.
    finally:
        await connection.close()  # 종료 때 DB 연결을 닫는다. DB 파일은 그대로 남는다.
        await vllm_client.close()  # 종료 때 요청 처리에 쓴 vLLM 연결풀을 닫는다.


app = FastAPI(
    title="노무, 전월세 상담 API(키 관리와 사용량 기록)",
    description="관리자가 발급한 서비스 키로 인증하고, 키별 토큰 사용량을 SQLite 에 기록합니다.",
    version="3.0.0",
    lifespan=lifespan,
)

register_error_handlers(app)
app.include_router(system_router)
app.include_router(admin_router)
app.include_router(metered_router)
```

### 1-3. 원격 노트북 준비

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01과 같은 준비 코드(경로, 서버 주소)와 노무 질문에, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 관리자 키를 담은 헤더를 더해 한 셀에 모았습니다. 관리자 키는 헤더에만 넣고 화면에 출력하지 않습니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값(교안 01의 준비 셀에 .env 읽기를 더함) ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

lesson_dir = Path(".").resolve()  # 이 노트북의 단원 폴더(정답 노트북은 "..")
day_dir = lesson_dir.parent                # data, images 가 있는 일차 폴더
data_dir = day_dir / "data"

# 원격 저장소의 .env를 읽습니다(아래에서 키를 읽을 때 씁니다)
load_dotenv(day_dir.parent / ".env", override=True)

SERVING_URL = "http://127.0.0.1:8080"   # 터미널 2에서 띄운 FastAPI 서빙 API

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기(교안 01과 같은 함수) ----
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(서빙 벤치마킹 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 노무 질문, 관리자 키, 요청 시간 제한 ----
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")

# 1-1에서 .env에 저장한 관리자 키를 관리자 주소에 보낼 헤더에 담습니다(키 값은 출력하지 않습니다)
ADMIN_API_KEY = os.environ["ADMIN_API_KEY"]
ADMIN_HEADERS = {"X-Admin-Key": ADMIN_API_KEY}

# 노트북이 FastAPI 응답을 기다리는 최대 시간(초). httpx 기본값 5초는 답 생성 시간보다 짧을 수 있습니다
REQUEST_TIMEOUT = 120

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상태 확인 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>는 키 없이 열어 둔 주소라, 서버가 요청을 받을 수 있는지 확인할 때 씁니다.</p>
</div>

In [ ]:
# 키 없이 상태 확인 주소를 부릅니다(vLLM은 부르지 않습니다)
health_response = httpx.get(f"{SERVING_URL}/healthz", timeout=REQUEST_TIMEOUT)
print(f"상태 코드: {health_response.status_code} | 본문: {health_response.json()}")

## 2. SQLite 키 저장소

### 2-1. SQLite 파일과 두 테이블

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02의 키 파일은 서버가 뜰 때 한 번 읽으므로, 키를 더하거나 끊으려면 파일을 고치고 서버를 다시 켜야 합니다. 키 원문도 파일에 그대로 적혀 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SQLite</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">따로 띄우는 DB 서버 없이 파일 하나에 테이블을 저장하는 데이터베이스입니다.</li>
<li style="margin:3px 0;">서버는 요청이 올 때마다 DB 파일을 읽습니다. 그래서 발급하거나 폐기한 키가 서버를 다시 켜지 않아도 바로 반영됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">aiosqlite:</strong> SQLite를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async</code>/<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 읽고 쓰게 해 주는 라이브러리입니다. 연결마다 스레드 하나가 파일 읽기와 쓰기를 맡아, DB를 쓰는 동안에도 서버가 다른 요청을 받습니다. SQLite는 파일 하나에 한 번에 한 곳만 쓰므로, 동시에 온 쓰기는 차례로 처리됩니다.</li>
</ul>
</div>

```python
# dependencies/db_security.py
def get_db(request: Request):
    """요청이 들어온 앱의 lifespan 이 연 DB 연결을 꺼낸다(app.state.db).

    엔드포인트와 의존성은 connection=Depends(get_db) 로 받아 key_store 함수에 넘긴다.
    """
    return request.app.state.db
```

```python
# repositories/key_store.py
# 서비스 키와 사용량을 저장할 SQLite 파일. 설정의 service_db_path(환경 변수 SERVICE_DB_PATH, 기본 output/service.db)
SERVICE_DB_PATH = settings.service_db_path


async def init_db(connection):
    """키 테이블(api_keys)과 사용량 테이블(usage)을 만든다. 이미 있으면 그대로 둔다."""
    # IF NOT EXISTS: 테이블이 이미 있으면 그대로 둬, 서버를 다시 켜도 발급한 키와 사용량 기록이 남는다
    await connection.executescript("""
        -- 서비스 키 테이블: 발급한 키 하나가 한 행
        CREATE TABLE IF NOT EXISTS api_keys (
            key_id     INTEGER PRIMARY KEY AUTOINCREMENT,  -- 키 번호. 발급할 때마다 1씩 저절로 늘어난다
            key_hash   TEXT NOT NULL UNIQUE,  -- 키 원문의 SHA-256 해시(원문은 저장하지 않는다)
            user       TEXT NOT NULL,         -- 키를 받은 사용자(앱, 고객사) 이름
            plan       TEXT NOT NULL,         -- 요금제(free, pro)
            created_at TEXT NOT NULL,         -- 발급 시각(UTC)
            revoked_at TEXT                   -- 폐기 시각. 폐기 전에는 비어 있다(NULL)
        );
        -- 사용량 테이블: 처리한 상담 요청 하나가 한 행
        CREATE TABLE IF NOT EXISTS usage (
            id                INTEGER PRIMARY KEY AUTOINCREMENT,  -- 행 번호(1부터 저절로 늘어난다)
            key_id            INTEGER NOT NULL,  -- 요청에 쓴 키(api_keys 의 key_id)
            path              TEXT NOT NULL,     -- 요청한 주소(/v1/labor/chat 등)
            prompt_tokens     INTEGER NOT NULL,  -- 입력 토큰 수(서버가 붙인 시스템 메시지 포함)
            cached_tokens     INTEGER NOT NULL,  -- 캐시 적중 토큰: 입력 토큰 가운데 캐시에서 다시 쓴 토큰 수
            completion_tokens INTEGER NOT NULL,  -- 출력 토큰 수(생각 포함)
            created_at        TEXT NOT NULL      -- 기록 시각(UTC)
        );
    """)
```

### 2-2. 키 원문 대신 해시 저장

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">DB 파일은 백업본이나 복사본에 섞여 밖으로 나가기도 하고, 서버에 접속할 수 있는 다른 사람이 가져갈 수도 있습니다. 키 원문을 그대로 저장하면 그 파일 하나로 노무 상담 앱, 전월세 상담 앱의 키를 모두 쓸 수 있습니다. 그래서 이 서버는 원문 대신, 원문으로 되돌릴 수 없는 해시를 저장합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">해시 함수(hash function)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">입력 문자열을 정해진 길이의 값으로 바꾸는 함수입니다. 같은 입력은 늘 같은 값이 되고, 값에서 입력을 거꾸로 계산하기는 사실상 불가능합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SHA-256:</strong> 이 교안이 쓰는 해시 함수입니다. 결과를 64자리 16진수 문자열로 씁니다. 파이썬 표준 라이브러리 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">hashlib</code>에 있습니다.</li>
</ul>
<p style="margin:6px 0;">먼저 노트북에서 키 모양의 문자열 세 개를 해시로 바꿔 봅니다. 첫째와 둘째는 같은 문자열이고, 셋째는 마지막 한 글자만 다릅니다.</p>
</div>

In [ ]:
import hashlib

# 키 모양의 예시 문자열 세 개를 SHA-256 해시로 바꿉니다
examples = ["svc-example-key-0001", "svc-example-key-0001", "svc-example-key-0002"]
hash_rows = []
for text in examples:
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    hash_rows.append({"입력": text, "SHA-256 해시": digest, "글자 수": len(digest)})

hash_table = pd.DataFrame(hash_rows)
display(hash_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫째, 둘째 행:</strong> 입력이 같아 해시도 같습니다. 서버는 요청에 담긴 키를 해시로 바꿔 저장된 해시와 비교하는 것으로 키를 찾습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">셋째 행:</strong> 한 글자만 달라도 해시 전체가 달라집니다. 그래서 키를 조금씩 바꿔 가며 해시가 비슷해지는지 보는 방법으로는 원문을 맞힐 수 없습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">글자 수 열:</strong> 입력 길이와 상관없이 늘 64자입니다.</li>
</ul>
<p style="margin:6px 0;">키 저장소의 함수도 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">hashlib.sha256</code>을 씁니다.</p>
</div>

```python
# repositories/key_store.py
def hash_key(api_key):
    """키 원문의 SHA-256 해시(64자리 16진수 문자열)를 돌려준다."""
    return hashlib.sha256(api_key.encode("utf-8")).hexdigest()


async def issue_key(connection, user, plan):
    """새 서비스 키를 만들어 해시만 저장하고, 키 원문이 든 딕셔너리를 돌려준다(원문은 이때 한 번만 보인다)."""
    # 1) 추측할 수 없는 무작위 키를 만든다
    api_key = "svc-" + secrets.token_urlsafe(24)  # 추측할 수 없는 긴 무작위 문자열

    # 2) 키 원문 대신 해시를 저장한다. key_id 는 DB 가 1, 2, 3 처럼 차례로 매긴다
    cursor = await connection.execute(
        "INSERT INTO api_keys (key_hash, user, plan, created_at) VALUES (?, ?, ?, ?)",
        (hash_key(api_key), user, plan, now_text()),
    )
    await connection.commit()
    key_id = cursor.lastrowid  # 방금 넣은 행의 번호. 비밀이 아니라 차례로 늘어나도 된다(비밀은 api_key)

    # 원문은 이 딕셔너리에만 담겨 발급 응답에서 한 번만 보인다(잃어버리면 새로 발급하고 옛 키는 폐기한다)
    return {"key_id": key_id, "api_key": api_key, "user": user, "plan": plan}


async def find_active_key(connection, api_key):
    """키 원문의 해시로 폐기되지 않은 키를 찾아 {"key_id", "user", "plan"} 을 돌려준다. 없으면 None."""
    # 해시가 같고 폐기되지 않은(revoked_at IS NULL) 행만 찾는다. 폐기한 키는 없는 키와 같아진다
    cursor = await connection.execute(
        "SELECT key_id, user, plan FROM api_keys WHERE key_hash = ? AND revoked_at IS NULL",
        (hash_key(api_key),),
    )
    row = await cursor.fetchone()
    if row is None:
        return None
    return {"key_id": row["key_id"], "user": row["user"], "plan": row["plan"]}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">DB 파일이 새어 나가면:</strong> 누가 파일을 읽어도 해시에서 원문을 계산할 수 없으므로, 그 파일만으로는 상담 주소를 부를 수 없습니다. 훔친 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_hash</code> 값을 그대로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code>에 넣어도 서버가 그 값을 한 번 더 해시로 바꿔 찾으므로 맞는 행이 없습니다. 이 방식이 안전한 것은 키가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secrets</code>로 만든 긴 무작위 문자열이기 때문입니다. 사람이 정하는 비밀번호는 짧고 흔한 값이 많아서, 흔한 비밀번호를 미리 해시해 맞춰 보면 찾아낼 수 있습니다. 그래서 비밀번호는 이렇게 저장하지 않습니다.</li>
</ul>
</div>

## 3. 키 발급과 사용

### 3-1. 관리자 주소와 관리자 키

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키를 발급하고 폐기하는 관리 기능은 서비스 키가 아닌 관리자 키로 따로 보호합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">따로 두는 이유:</strong> 서비스 키는 API를 쓰는 권한이고, 관리자 키는 그 권한을 나눠 주고 거두는 권한입니다. 한 키로 둘 다 허용하면 키를 받은 사용자가 키를 더 만들거나 다른 사용자의 키를 폐기할 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실제 서비스:</strong> 회원이 로그인해 자기 키를 직접 만드는 서비스도 다른 사용자의 키 폐기, 요금제 결정, 전체 사용량 조회는 관리자만 합니다. OpenAI도 모델을 부르는 키와 별도로 조직 관리용 Admin API key를 둡니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">발급 방식:</strong> 회원이 직접 발급하는 방식과, 관리자가 발급해 전달하는 방식(사내 앱, 계약한 고객사)이 있습니다. 이 교안은 관리자 발급 방식입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">403 Forbidden</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청한 쪽이 누구인지는 알지만, 그 기능을 쓸 권한이 없다는 상태 코드입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">401과 다른 점:</strong> 401은 요청한 쪽이 누구인지 확인하지 못했다는 뜻(인증 실패)이고, 403은 권한이 없다는 뜻(권한 없음)입니다.</li>
</ul>
</div>

```python
# routers/admin_router.py
# 관리자 키는 서버 쪽 .env 에만 둔다(설정의 admin_api_key, 환경 변수 ADMIN_API_KEY)
ADMIN_API_KEY = settings.admin_api_key


admin_key_header = APIKeyHeader(name="X-Admin-Key", auto_error=False)


def require_admin(admin_key: str | None = Depends(admin_key_header)):
    """X-Admin-Key 가 서버의 관리자 키와 같은지 확인한다. 다르면 403."""
    # compare_digest: 글자를 앞에서부터 비교하다 멈추지 않아, 걸린 시간으로 키를 추측할 수 없다
    if admin_key is None or not secrets.compare_digest(admin_key, ADMIN_API_KEY):
        raise HTTPException(status_code=403, detail="관리자 키가 필요합니다. X-Admin-Key 헤더를 확인하세요.")


class IssueKeyRequest(BaseModel):
    """키 발급 요청."""

    user: str = Field(min_length=1, max_length=50, description="키를 받을 사용자(앱, 고객사) 이름")
    # 요금제는 두 값만 받는다. 그 밖의 값은 교안 02 의 오류 양식과 같은 422(invalid_request)
    plan: Literal["free", "pro"] = Field(description="요금제. free 는 60초에 3번, pro 는 30번")


class IssuedKey(BaseModel):
    """발급한 키. api_key 원문은 이 응답에서 한 번만 보인다."""

    key_id: int
    api_key: str
    user: str
    plan: str


# 라우터에 붙인 의존성은 이 묶음의 모든 주소에 적용된다
# 교안 02 는 앱에 붙일 때(include_router) 지정했다. 라우터를 만들 때 지정하면 어느 앱에 넣어도 확인이 빠지지 않는다
admin_router = APIRouter(prefix="/v1/admin/keys", tags=["키 관리(관리자)"], dependencies=[Depends(require_admin)])


# 새 키를 만드는 주소라 200 대신 201 Created(새 것을 만들었다)로 답한다
@admin_router.post("", status_code=201, response_model=IssuedKey)
async def create_key(request: IssueKeyRequest, connection=Depends(get_db)):
    """새 서비스 키를 발급한다(201 Created)."""
    return await issue_key(connection, request.user, request.plan)
```

### 3-2. 관리자 키 없이 한 번, 관리자 키로 한 번 발급 요청

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습 키는 &quot;노무 상담 앱&quot;에 pro 요금제로 발급합니다. 이 절부터 5절까지 상담 요청을 여러 번 보내므로 60초에 30번까지 받는 pro 요금제를 씁니다. 먼저 관리자 키 헤더 없이 보냅니다.</p>
</div>

In [ ]:
# 관리자 키 헤더 없이 키 발급을 요청합니다
issue_body = {"user": "노무 상담 앱", "plan": "pro"}
no_admin_response = httpx.post(f"{SERVING_URL}/v1/admin/keys", json=issue_body, timeout=REQUEST_TIMEOUT)
print(f"상태 코드: {no_admin_response.status_code} | 본문: {no_admin_response.json()}")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">403과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">admin_only</code>:</strong> 관리자 주소 묶음에 붙인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_admin</code>이 엔드포인트보다 먼저 실행되어 거절했습니다. 키는 만들어지지 않았습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">404와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">not_found</code>가 나오면:</strong> 터미널 2에 아직 앞 노트북의 앱(교안 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code> 또는 과제 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.secure_main</code>)이 떠 있는 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 그 앱을 끄고 1-1의 폴더 이동 블록과 1-2의 명령으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.managed_main</code>을 띄웁니다.</li>
</ul>
<p style="margin:6px 0;">같은 본문을 준비 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ADMIN_HEADERS</code>와 함께 보냅니다.</p>
</div>

In [ ]:
# 같은 본문을 관리자 키 헤더와 함께 보내 pro 요금제 키를 발급합니다
issue_response = httpx.post(f"{SERVING_URL}/v1/admin/keys", headers=ADMIN_HEADERS, json=issue_body, timeout=REQUEST_TIMEOUT)
issued = issue_response.json()

# 상태 코드와 발급 응답을 함께 봅니다
issued_view = {"상태 코드": issue_response.status_code, **issued}
display(issued_view)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">201:</strong> 키를 새로 만들었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">api_key</code>:</strong> 키 원문입니다. 서버는 이 원문을 저장하지 않으므로 원문을 볼 수 있는 곳은 이 응답 하나입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">화면에 보이는 키 원문:</strong> 이 노트북은 실행 중에 만든 연습용 키라 화면에 보이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">issued</code> 변수에 담아 이어 씁니다. 실제 서비스에서는 키를 받을 사람에게만 전합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">key_id</code>:</strong> 관리자가 이 키를 가리킬 때 쓰는 번호입니다. 5절의 폐기와 4절의 관리자 사용량 조회 주소에 들어갑니다. 비밀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key</code> 원문이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>는 비밀이 아니라서, 1, 2, 3처럼 차례로 늘어나는 번호여도 됩니다.</li>
</ul>
</div>

### 3-3. 발급한 키로 상담 주소 호출

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상담 주소의 키 확인과 요청 한도는 교안 02와 함수 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_service_key</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>)과 흐름이 같고, 파일만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/db_security.py</code>로 다릅니다. 달라진 점은 키를 JSON 파일 대신 DB에서 해시로 찾는다는 것입니다.</p>
</div>

```python
# dependencies/db_security.py
# 요금제별 60초당 요청 한도
PLAN_LIMITS = {"free": 3, "pro": 30}


async def require_service_key(api_key: str | None = Depends(api_key_header), connection=Depends(get_db)):
    """X-API-Key 의 키를 DB 에서 찾아 {"key_id", "user", "plan"} 을 돌려준다. 없거나 폐기된 키면 401."""
    if api_key is None:
        raise HTTPException(status_code=401, detail="서비스 키가 없습니다. X-API-Key 헤더를 확인하세요.")

    client = await find_active_key(connection, api_key)  # 폐기한 키도 찾지 못해 없는 키와 같이 401 이 된다
    if client is None:
        raise HTTPException(status_code=401, detail="발급되지 않았거나 폐기된 서비스 키입니다.")
    return client


async def check_rate_limit(client: dict = Depends(require_service_key)):
    """DB 키 확인을 먼저 거친 뒤, key_id 의 최근 60초 요청 수가 요금제 한도 안이면 client 를 돌려준다. 넘으면 429."""
    # 메모리의 기록만 고치는 함수라 async def 로 둔다(한 이벤트 루프에서 차례로 실행되어 기록이 엉키지 않는다)
    limit = PLAN_LIMITS[client["plan"]]
    now = time.monotonic()

    # 1) 60초 창 안의 요청 시각만 남긴다(교안 02 의 check_rate_limit 과 같은 계산)
    # 교안 02 는 키 원문별로 기록했다. 원문을 메모리에도 두지 않으려고 key_id 별로 기록한다
    recent = []
    for sent_at in request_log.get(client["key_id"], []):
        if now - sent_at < WINDOW_SECONDS:
            recent.append(sent_at)

    # 2) 한도를 다 썼으면 가장 오래된 요청이 창을 벗어날 때까지 남은 초를 Retry-After 로 알린다
    if len(recent) >= limit:
        retry_after = math.ceil(WINDOW_SECONDS - (now - recent[0]))
        raise HTTPException(
            status_code=429,
            detail=f"{client['plan']} 요금제는 {WINDOW_SECONDS}초에 {limit}번까지 요청할 수 있습니다.",
            headers={"Retry-After": str(retry_after)},
        )

    # 3) 통과한 요청만 기록한다
    recent.append(now)
    request_log[client["key_id"]] = recent
    return client
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02 키 파일의 pro 키와 방금 발급한 키로 같은 노무 질문을 보냅니다.</p>
</div>

In [ ]:
# 발급한 키를 X-API-Key 헤더에 담습니다
issued_headers = {"X-API-Key": issued["api_key"]}

# 교안 02 키 파일의 pro 키와 발급한 키로 같은 노무 질문을 보냅니다
labor_question = labor_questions[0]["question"]
key_cases = {
    "교안 02 키 파일의 pro 키": {"X-API-Key": "svc-pro-demo-0002"},
    "발급한 키": issued_headers,
}

key_rows = []
for case, headers in key_cases.items():
    response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers=headers, json={"question": labor_question}, timeout=REQUEST_TIMEOUT)

    # 성공하면 모델의 답(테이블이 넓어지지 않게 앞 80자), 실패하면 error 항목의 code 와 message 를 본문 칸에 넣습니다
    body = response.json()
    if response.status_code == 200:
        summary = body["answer"][:80]
    else:
        summary = f"{body['error']['code']}: {body['error']['message']}"
    key_rows.append({"경우": case, "상태 코드": response.status_code, "본문": summary})

key_table = pd.DataFrame(key_rows)
display(key_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 02 키 파일의 pro 키 행의 401과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">invalid_service_key</code>:</strong> 이 앱은 키를 DB에서만 찾습니다. 키 파일의 연습용 키는 DB에 없으므로 받지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">발급한 키 행의 200:</strong> 키의 해시가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_keys</code> 테이블에 있고 폐기되지 않아 노무 답이 왔습니다.</li>
</ul>
</div>

### 3-4. DB 파일에 저장된 값 확인

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버가 이 교안 폴더에 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/service.db</code>를 노트북에서 열어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_keys</code> 테이블을 읽습니다. 조회(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SELECT</code>)만 하고 쓰지 않습니다.</p>
</div>

In [ ]:
import sqlite3

# 서버가 이 교안 폴더에 만든 DB 파일을 열어 api_keys 테이블을 읽습니다(조회만 합니다)
db_path = lesson_dir / "output" / "service.db"
connection = sqlite3.connect(db_path)
stored_keys = pd.read_sql_query("SELECT key_id, user, plan, key_hash, revoked_at FROM api_keys", connection)  # 조회 결과를 바로 DataFrame 으로 받습니다
connection.close()

# 발급 응답의 키 원문을 해시로 바꿔 저장된 해시와 같은지 비교합니다
issued_hash = hashlib.sha256(issued["api_key"].encode("utf-8")).hexdigest()
stored_keys["해시 글자 수"] = stored_keys["key_hash"].str.len()
stored_keys["발급한 키의 해시와 같음"] = stored_keys["key_hash"] == issued_hash
display(stored_keys)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">key_hash</code> 열:</strong> 테이블에는 키 원문 열이 없고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_hash</code>만 있으며, 해시 글자 수는 64입니다. 3-2의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key</code> 원문은 DB 어디에도 없습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">발급한 키의 해시와 같음 열:</strong> 3-2에서 발급한 키의 행만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">True</code>입니다. 서버도 요청의 키를 같은 방식으로 해시로 바꿔 이 행을 찾습니다.</li>
<li style="margin:3px 0;">발급 셀을 여러 번 실행했다면 실행한 만큼 행이 있고, 마지막에 발급한 키의 행만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">True</code>입니다.</li>
</ul>
</div>

## 4. 사용량 기록과 조회

### 4-1. 엔드포인트 안에서 기록하기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">토큰 사용량은 vLLM의 답이 나온 뒤에야 알 수 있습니다. 교안 02처럼 앱에 붙일 때 더한 의존성은 엔드포인트보다 먼저 실행되어 답을 볼 수 없으므로, 엔드포인트 안에서 답을 받은 뒤 기록합니다.</p>
</div>

```python
# routers/metered_router.py
metered_router = APIRouter(prefix="/v1", tags=["상담(사용량 기록)"])


# 창구 이름 -> 시스템 메시지. 주소의 desk 값으로 고른다
SYSTEM_PROMPTS = {"labor": LABOR_SYSTEM_PROMPT, "lease": LEASE_SYSTEM_PROMPT}


# 창구 이름 -> 생각 상한. 노무 창구는 시스템 메시지가 길어 더 작은 값을 쓴다(설정 max_tokens_think_labor)
THINK_LIMITS = {"labor": settings.max_tokens_think_labor, "lease": settings.max_tokens_think}


# 경로 매개변수 desk 에 Literal 을 붙여 노무, 전월세 창구를 엔드포인트 하나로 처리한다
@metered_router.post("/{desk}/chat", response_model=ChatResponse)
async def metered_chat(
    desk: Literal["labor", "lease"],
    request: ChatRequest,
    client: dict = Depends(check_rate_limit),
    connection=Depends(get_db),  # lifespan 이 연 DB 연결
):
    """상담 질문 하나의 답을 돌려주고, 그 요청의 토큰 사용량을 키별로 기록한다."""
    # 1) desk 에 맞는 시스템 메시지로 vLLM 에 묻는다(desk 가 labor, lease 가 아니면 422)
    completion = await ask_vllm(SYSTEM_PROMPTS[desk], request.question, request.think, max_tokens_think=THINK_LIMITS[desk])

    # 2) 그 요청의 토큰 사용량을 기록한다. client 는 check_rate_limit 을 통과한 키다
    usage = completion.usage
    await record_usage(
        connection, client["key_id"], f"/v1/{desk}/chat",
        usage.prompt_tokens, usage.prompt_tokens_details.cached_tokens, usage.completion_tokens,
    )
    return to_chat_response(completion)
```

```python
# repositories/key_store.py
async def record_usage(connection, key_id, path, prompt_tokens, cached_tokens, completion_tokens):
    """요청 하나의 토큰 사용량을 usage 테이블에 한 행으로 남긴다."""
    await connection.execute(
        "INSERT INTO usage (key_id, path, prompt_tokens, cached_tokens, completion_tokens, created_at) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (key_id, path, prompt_tokens, cached_tokens, completion_tokens, now_text()),
    )
    await connection.commit()
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실패한 요청:</strong> vLLM이 실패해 502, 504가 되면 기록 줄에 닿기 전에 예외가 나므로 사용량 행이 생기지 않습니다. 401, 429, 422로 거절한 요청도 엔드포인트가 실행되지 않아 사용량 행이 생기지 않습니다. 다만 본문 양식이나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">desk</code> 값이 틀려 422가 된 요청은 교안 02와 같이 키 확인과 요청 한도 확인을 먼저 통과했으므로 요청 한도는 1번 씁니다.</li>
</ul>
</div>

### 4-2. 스트리밍 주소의 기록과 사용량 조회 주소

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">스트리밍 주소는 토큰 수가 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 이벤트에 옵니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_and_record</code>는 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_stream</code>이 만든 SSE 줄을 바꾸지 않고 보내면서, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 줄을 만나면 그 토큰 수를 기록합니다.</p>
</div>

```python
# routers/metered_router.py
async def relay_and_record(stream, connection, key_id, path):
    """relay_stream 이 만든 SSE 줄을 그대로 보내면서, usage 이벤트를 만나면 사용량을 기록한다."""
    # usage 이벤트 전에 스트림이 끊기면 기록하지 않는다
    async for line in relay_stream(stream):
        # usage 이벤트 줄: data: {"type": "usage", ...}
        if line.startswith('data: {"type": "usage"'):
            usage = json.loads(line[len("data: "):])
            await record_usage(connection, key_id, path, usage["prompt_tokens"], usage["cached_tokens"], usage["completion_tokens"])
        yield line


@metered_router.post("/{desk}/chat/stream")
async def metered_chat_stream(
    desk: Literal["labor", "lease"],
    request: ChatRequest,
    client: dict = Depends(check_rate_limit),
    connection=Depends(get_db),
):
    """상담 답을 SSE 로 중계하고, 마지막 usage 이벤트의 토큰 수를 키별로 기록한다."""
    stream = await open_vllm_stream(SYSTEM_PROMPTS[desk], request.question, request.think, max_tokens_think=THINK_LIMITS[desk])
    return StreamingResponse(
        relay_and_record(stream, connection, client["key_id"], f"/v1/{desk}/chat/stream"),
        media_type="text/event-stream",
    )


# 서비스 키 확인만 붙여 요청 한도에 세지 않는다
@metered_router.get("/usage")
async def my_usage(
    start: date | None = None,  # 쿼리 매개변수 ?start=2026-11-01. 날짜 모양이 아니면 422
    end: date | None = None,  # 날짜는 UTC 기준(한국 시간 오전 9시 전 요청은 전날로 잡힌다)
    client: dict = Depends(require_service_key),
    connection=Depends(get_db),
):
    """X-API-Key 주인의 사용량 합계를 돌려준다(자기 키의 것만 볼 수 있다). start, end 를 주면 그 기간만 더한다."""
    return await usage_summary(connection, client["key_id"], start, end)
```

```python
# repositories/key_store.py
async def usage_summary(connection, key_id, start=None, end=None):
    """키 하나의 요청 수와 토큰 합계를 딕셔너리로 돌려준다. start, end(날짜, 둘 다 포함)를 주면 그 기간만 더한다."""
    # 1) 조건을 모은다. created_at 은 '2026-11-08T03:00:00+00:00' 모양 UTC 문자열이라 날짜 문자열과 크기 비교가 된다
    conditions = ["key_id = ?"]
    values = [key_id]
    if start is not None:
        conditions.append("created_at >= ?")
        values.append(start.isoformat())
    if end is not None:
        conditions.append("created_at < ?")  # 끝 날짜를 포함하려고 다음 날 0시 전까지 더한다
        values.append((end + timedelta(days=1)).isoformat())

    # 2) 기록이 없으면 SUM 이 NULL 이므로 COALESCE 로 0 으로 바꾼다
    cursor = await connection.execute(
        "SELECT COUNT(*) AS requests, "
        "COALESCE(SUM(prompt_tokens), 0) AS prompt_tokens, "
        "COALESCE(SUM(cached_tokens), 0) AS cached_tokens, "
        "COALESCE(SUM(completion_tokens), 0) AS completion_tokens "
        "FROM usage WHERE " + " AND ".join(conditions),
        values,
    )
    row = await cursor.fetchone()
    return {
        "key_id": key_id,
        "start": start,
        "end": end,
        "requests": row["requests"],
        "prompt_tokens": row["prompt_tokens"],
        "cached_tokens": row["cached_tokens"],
        "completion_tokens": row["completion_tokens"],
        "total_tokens": row["prompt_tokens"] + row["completion_tokens"],
    }
```

```python
# routers/admin_router.py
# 관리자 키 확인은 admin_router 에 붙인 require_admin 이 이 함수보다 먼저 한다(키가 없거나 틀리면 403)
@admin_router.get("/{key_id}/usage", response_model=UsageSummary)
async def read_key_usage(
    key_id: int,
    start: date | None = None,  # 쿼리 매개변수 ?start=2026-11-01. 날짜 모양이 아니면 422
    end: date | None = None,  # 날짜는 UTC 기준(한국 시간 오전 9시 전 요청은 전날로 잡힌다)
    connection=Depends(get_db),
):
    """키 하나의 요청 수와 토큰 합계를 돌려준다. start, end 를 주면 그 기간만 더한다."""
    return await usage_summary(connection, key_id, start, end)  # 관리자는 아무 키나 본다. /v1/usage 와 같은 함수로 계산한다
```

### 4-3. 일반 주소와 스트리밍 주소로 요청 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">발급한 키로 노무 창구의 일반 주소와 스트리밍 주소에 질문을 하나씩 보냅니다.</p>
</div>

In [ ]:
# 노무 창구의 일반 주소에 질문을 보내고 응답의 usage 를 꺼냅니다
chat_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers=issued_headers, json={"question": labor_questions[1]["question"]}, timeout=REQUEST_TIMEOUT)
chat_usage = chat_response.json()["usage"]

# 같은 키로 스트리밍 주소에 질문을 보내고, 교안 01 6절과 같이 한 줄씩 읽어 usage 이벤트를 꺼냅니다
stream_usage = None
with httpx.stream("POST", f"{SERVING_URL}/v1/labor/chat/stream", headers=issued_headers, json={"question": labor_questions[2]["question"]}, timeout=REQUEST_TIMEOUT) as response:
    stream_status = response.status_code
    for line in response.iter_lines():
        if line.startswith('data: {"type": "usage"'):
            stream_usage = json.loads(line.removeprefix("data: "))

# 두 요청의 상태 코드와 토큰 수를 한 테이블로 모읍니다
metered_rows = [
    {"주소": "/v1/labor/chat", "상태 코드": chat_response.status_code, "prompt_tokens": chat_usage["prompt_tokens"], "cached_tokens": chat_usage["cached_tokens"], "completion_tokens": chat_usage["completion_tokens"]},
    {"주소": "/v1/labor/chat/stream", "상태 코드": stream_status, "prompt_tokens": stream_usage["prompt_tokens"], "cached_tokens": stream_usage["cached_tokens"], "completion_tokens": stream_usage["completion_tokens"]},
]
metered_table = pd.DataFrame(metered_rows)
display(metered_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">두 상담 주소의 200과 토큰 수:</strong> 일반 응답은 본문의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>, 스트리밍 응답은 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 이벤트에 토큰 수가 있고, 서버는 같은 값을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 테이블에 한 행씩 기록했습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code> 열:</strong> 질문만이 아니라 서버가 붙인 시스템 메시지(참고 조문)까지 센 값이라(교안 01 4-3), 질문이 한 줄이어도 수천 개입니다. 같은 질문이면 실행할 때마다 같고, 답의 길이인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion_tokens</code>만 실행마다 달라집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">cached_tokens</code> 열:</strong> 서빙 벤치마킹 단원의 접두어 캐싱에서 본 캐시 적중 토큰입니다. 3-3에서 같은 노무 시스템 메시지를 이미 보냈으므로 두 행 모두 0보다 크고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code>보다는 작습니다.</li>
</ul>
<p style="margin:6px 0;">이 키의 사용량은 대부분 서버가 붙인 시스템 메시지입니다. 그래서 요금을 정할 때는 이 몫을 누가 낼지도 함께 정해야 합니다. 캐시 적중 토큰은 계산을 다시 하지 않은 몫이라, 실제 AI 서비스는 이 몫의 입력 요금을 낮게 받기도 합니다.</p>
</div>

### 4-4. 사용자와 관리자의 사용량 조회

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 키의 사용량을 서비스 키로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/usage</code>에서, 관리자 키로 관리자 주소에서 각각 조회합니다.</p>
</div>

In [ ]:
# 발급한 키로 내 사용량을 조회합니다(요청 한도에 세지 않는 주소입니다)
my_usage = httpx.get(f"{SERVING_URL}/v1/usage", headers=issued_headers, timeout=REQUEST_TIMEOUT).json()

# 관리자 키로 같은 키의 사용량을 조회합니다
admin_usage = httpx.get(f"{SERVING_URL}/v1/admin/keys/{issued['key_id']}/usage", headers=ADMIN_HEADERS, timeout=REQUEST_TIMEOUT).json()

# 두 조회 결과를 한 테이블로 나란히 봅니다
usage_table = pd.DataFrame([
    {"조회한 곳": "GET /v1/usage(서비스 키)", **my_usage},
    {"조회한 곳": "GET /v1/admin/keys/{key_id}/usage(관리자 키)", **admin_usage},
])
display(usage_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">두 행의 값:</strong> 서로 같습니다. 두 주소가 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 테이블에서 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>의 행을 더했기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">requests</code>:</strong> 이 키로 200을 받은 상담 요청 수입니다. 3-3의 노무 질문 하나와 4-3의 일반, 스트리밍 요청 둘을 더한 값이고, 셀을 다시 실행했다면 그만큼 늘어 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">total_tokens</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion_tokens</code>의 합입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cached_tokens</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code> 안에 든 몫이라 따로 더하지 않습니다.</li>
</ul>
<p style="margin:6px 0;">같은 키를 기간을 정해 다시 조회합니다. 두 사용량 주소는 쿼리 매개변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">start</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">end</code>(두 날짜 모두 포함)를 받습니다.</p>
</div>

In [ ]:
from datetime import datetime, timedelta, timezone

# 서버가 created_at 을 UTC 로 저장하므로 날짜도 UTC 기준으로 정합니다
today = datetime.now(timezone.utc).date()
yesterday = today - timedelta(days=1)

# 같은 키의 사용량을 오늘 하루, 어제 하루로 나눠 관리자 주소에서 조회합니다
period_rows = []
for label, day in {"오늘": today, "어제": yesterday}.items():
    period = {"start": day.isoformat(), "end": day.isoformat()}  # 2026-11-08 모양 날짜 문자열
    response = httpx.get(f"{SERVING_URL}/v1/admin/keys/{issued['key_id']}/usage", headers=ADMIN_HEADERS, params=period, timeout=REQUEST_TIMEOUT)
    period_rows.append({"기간": label, **response.json()})

period_table = pd.DataFrame(period_rows)
display(period_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">오늘 행:</strong> 위 표의 합계와 같습니다. 이 노트북에서 보낸 요청은 모두 오늘(UTC) 기록입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">어제 행:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests</code>와 토큰이 모두 0입니다. 기간을 주면 그 기간의 기록만 더합니다. 실무에서는 한 달 단위 기간으로 조회해 요금을 청구합니다.</li>
</ul>
</div>

## 5. 키 폐기

### 5-1. 행을 지우지 않는 폐기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키가 유출되었거나 계약이 끝난 사용자의 키는 더 받지 않아야 합니다. 이 서버는 키 행을 지우지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">revoked_at</code>에 폐기 시각을 채웁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">DELETE 메서드</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">대상을 지우거나 끊는 요청에 쓰는 HTTP 메서드입니다. 이 서버는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DELETE /v1/admin/keys/{key_id}</code>를 키 폐기 주소로 씁니다.</li>
<li style="margin:3px 0;">키를 쓰는 쪽에서 보면 키가 없어지므로 주소는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DELETE</code>를 씁니다. 하지만 DB 안에서는 행을 지우지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">revoked_at</code>만 채워 기록을 남깁니다. 이렇게 지운 것으로 표시만 하는 방식을 <strong style="font-size:1.13em;color:#1f3a5f;">소프트 삭제(soft delete)</strong>라고 합니다.</li>
</ul>
</div>

```python
# dependencies/db_security.py
def get_db(request: Request):
    """요청이 들어온 앱의 lifespan 이 연 DB 연결을 꺼낸다(app.state.db).

    엔드포인트와 의존성은 connection=Depends(get_db) 로 받아 key_store 함수에 넘긴다.
    """
    return request.app.state.db
```

```python
# repositories/key_store.py
async def revoke_key(connection, key_id):
    """키를 폐기한다(행을 지우지 않고 revoked_at 을 채운다). 폐기했으면 True, 없거나 이미 폐기된 키면 False."""
    cursor = await connection.execute(
        "UPDATE api_keys SET revoked_at = ? WHERE key_id = ? AND revoked_at IS NULL",
        (now_text(), key_id),
    )
    await connection.commit()
    # 바뀐 행이 하나면 이번에 폐기한 것이다
    return cursor.rowcount == 1
```

```python
# routers/admin_router.py
# 관리자 키 확인은 admin_router 에 붙인 require_admin 이 이 함수보다 먼저 한다(키가 없거나 틀리면 403)
@admin_router.delete("/{key_id}")
async def delete_key(key_id: int, connection=Depends(get_db)):  # key_id: 경로의 키 번호. 숫자가 아니면 422
    """키를 폐기한다. 없거나 이미 폐기된 키면 404."""
    if not await revoke_key(connection, key_id):  # 바뀐 행이 없으면 없는 key_id 이거나 이미 폐기한 키다
        raise HTTPException(status_code=404, detail=f"폐기할 키가 없습니다: {key_id}")
    return {"key_id": key_id, "revoked": True}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키 행을 지우면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 테이블의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>가 누구의 키였는지 알 수 없게 됩니다. 행이 남아 있어야 폐기한 뒤에도 그 키의 사용량 합계와 사용자, 요금제를 조회해 청구와 점검에 쓸 수 있습니다.</li>
</ul>
</div>

### 5-2. 폐기하고 같은 키로 다시 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">관리자 키로 발급한 키를 폐기하고, 같은 키로 노무 질문을 보낸 뒤, 같은 키를 한 번 더 폐기합니다.</p>
</div>

In [ ]:
# 관리자 키로 발급한 키를 폐기합니다
revoke_url = f"{SERVING_URL}/v1/admin/keys/{issued['key_id']}"
revoke_response = httpx.delete(revoke_url, headers=ADMIN_HEADERS, timeout=REQUEST_TIMEOUT)

# 폐기한 키로 노무 질문을 보냅니다
revoked_chat_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers=issued_headers, json={"question": labor_question}, timeout=REQUEST_TIMEOUT)

# 같은 키를 한 번 더 폐기합니다
revoke_again_response = httpx.delete(revoke_url, headers=ADMIN_HEADERS, timeout=REQUEST_TIMEOUT)

# 세 단계의 상태 코드와 본문을 한 테이블로 모읍니다
revoke_cases = {"폐기": revoke_response, "폐기한 키로 상담": revoked_chat_response, "같은 키 다시 폐기": revoke_again_response}
revoke_rows = []
for step, response in revoke_cases.items():
    revoke_rows.append({"단계": step, "상태 코드": response.status_code, "본문": response.json()})

revoke_table = pd.DataFrame(revoke_rows)
display(revoke_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">폐기 행의 200:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">revoked_at</code>을 채운 행이 하나라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;revoked&quot;: True</code>로 답했습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">폐기한 키로 상담 행의 401과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">invalid_service_key</code>:</strong> 3-3에서 200을 받은 같은 키입니다. 서버를 다시 켜지 않아도 다음 요청부터 막혔습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">같은 키 다시 폐기 행의 404와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">not_found</code>:</strong> 이미 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">revoked_at</code>이 채워져 바뀐 행이 없습니다.</li>
</ul>
</div>

### 5-3. 폐기 뒤에도 남는 키 행과 사용량 기록

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">관리자 키로 키 목록을 조회합니다. 목록에는 키 원문과 해시가 없습니다.</p>
</div>

In [ ]:
# 관리자 키로 발급한 키 목록을 조회합니다
keys_response = httpx.get(f"{SERVING_URL}/v1/admin/keys", headers=ADMIN_HEADERS, timeout=REQUEST_TIMEOUT)
keys_table = pd.DataFrame(keys_response.json())
display(keys_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">revoked_at</code> 열:</strong> 폐기한 키의 행에는 폐기 시각(UTC, 세계 표준시)이 있습니다. 폐기하지 않은 키가 있다면 그 행은 비어 있습니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>). 폐기한 키의 행도 지워지지 않았습니다.</li>
</ul>
<p style="margin:6px 0;">폐기한 키의 사용량을 관리자 주소와 그 키의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/usage</code>로 조회합니다.</p>
</div>

In [ ]:
# 폐기한 키의 사용량을 관리자 주소와, 그 키로 부른 /v1/usage 에서 조회합니다
after_revoke_cases = {
    "관리자 주소(관리자 키)": httpx.get(f"{SERVING_URL}/v1/admin/keys/{issued['key_id']}/usage", headers=ADMIN_HEADERS, timeout=REQUEST_TIMEOUT),
    "/v1/usage(폐기한 키)": httpx.get(f"{SERVING_URL}/v1/usage", headers=issued_headers, timeout=REQUEST_TIMEOUT),
}

# 두 조회의 상태 코드와 본문을 한 테이블로 모읍니다
after_revoke_rows = []
for case, response in after_revoke_cases.items():
    after_revoke_rows.append({"조회한 곳": case, "상태 코드": response.status_code, "본문": response.json()})

after_revoke_table = pd.DataFrame(after_revoke_rows)
display(after_revoke_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">관리자 주소 행의 200:</strong> 4-4의 합계와 같은 값입니다. 폐기한 뒤 보낸 요청은 401이라 늘지 않았습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">/v1/usage 행의 401:</strong> 폐기한 키로는 자기 사용량도 볼 수 없습니다. 폐기한 키의 기록은 관리자만 조회할 수 있습니다.</li>
</ul>
</div>

## 6. 🖐️ 함께 따라하기: 전월세 상담 앱의 키 발급부터 폐기까지

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습에서 노무 상담 앱 키로 해 본 발급, 사용, 사용량 조회, 폐기를 이번에는 전월세 상담 앱 키로 직접 작성합니다. 서버 코드는 고치지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">사용자와 요금제:</strong> &quot;전월세 상담 앱&quot;에 free 요금제(60초에 3번) 키를 발급합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">desk</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease</code>인 전월세 창구 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/lease/chat</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_questions.jsonl</code>의 전월세 질문입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code> 줄을 바로 위 주석이 말하는 코드로 바꿉니다. 값 하나만 채우는 것이 아니라 보낼 방식(POST, GET, DELETE)과 주소, 헤더, 본문, 결과를 담을 변수까지 한 줄을 통째로 씁니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_03_정답.ipynb</code>의 같은 절에 있습니다.</p>
</div>

### 6-1. 전월세 상담 앱 키 발급

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 3-2를 참고합니다.
세부구현: 발급은 httpx.post로 SERVING_URL 뒤에 관리자 키 주소(/v1/admin/keys)를 붙인 f-문자열에 보냅니다. 관리자 주소에는 준비 셀의 관리자 키 헤더 딕셔너리를 보내고, 본문은 user에 사용자 이름, plan에 요금제 이름을 담은 딕셔너리입니다. timeout은 준비 셀의 REQUEST_TIMEOUT입니다.
```

</details>

In [ ]:
# 🖐️ 함께 따라하기: 관리자 키 헤더를 붙여 키 발급 주소에 POST 요청을 보내고, 응답을 lease_issue_response 에 담습니다
# 발급 요청 본문은 사용자 "전월세 상담 앱", 요금제 free 입니다
lease_issue_response = ...
lease_issued = lease_issue_response.json()

# 상태 코드와 발급 응답을 함께 봅니다(키 원문은 이 응답에서 한 번만 보입니다)
lease_issued_view = {"상태 코드": lease_issue_response.status_code, **lease_issued}
display(lease_issued_view)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 상태 코드가 201이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">전월세 상담 앱</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">plan</code>이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">free</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">svc-</code>로 시작하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>는 정수입니다.</p>
</div>

### 6-2. 전월세 질문 보내고 사용량 조회

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 3-3과 4-4를 참고합니다.
세부구현: (1) 헤더는 서비스 키 헤더 이름에 6-1 발급 응답(lease_issued)의 키 원문을 값으로 넣은 딕셔너리입니다.
(2) httpx.post로 SERVING_URL 뒤에 desk가 lease인 일반 상담 주소를 붙인 f-문자열에 보냅니다. 헤더는 (1)의 lease_headers, 본문은 question 항목에 첫 번째 전월세 질문을 담은 딕셔너리, timeout은 REQUEST_TIMEOUT입니다.
사용량 조회 셀: 서비스 키로 부르는 내 사용량 주소에 httpx.get으로 보내고, 응답 본문을 .json()으로 꺼냅니다.
```

</details>

In [ ]:
# 전월세 질문을 읽습니다
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")

# 🖐️ 함께 따라하기 (1): 6-1에서 발급한 키 원문을 서비스 키 헤더에 담은 딕셔너리를 lease_headers 에 담습니다
lease_headers = ...

# 🖐️ 함께 따라하기 (2): 그 헤더로 첫 번째 전월세 질문을 전월세 창구의 일반 상담 주소에 POST 요청으로 보내고, 응답을 lease_response 에 담습니다
lease_response = ...

# 상태 코드, 답, 토큰 수를 함께 봅니다
lease_body = lease_response.json()
lease_view = {"상태 코드": lease_response.status_code, "답": lease_body["answer"], "usage": lease_body["usage"]}
display(lease_view)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 상태 코드가 200이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">답</code>에 전월세 답이 있습니다.</p>
</div>

In [ ]:
# 🖐️ 함께 따라하기: 서비스 키 헤더(lease_headers)로 내 사용량 주소에 GET 요청을 보내고, 응답 본문(JSON)을 lease_usage 에 담습니다
lease_usage = ...
display(lease_usage)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>가 6-1의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>와 같고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests</code>가 1입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prompt_tokens</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cached_tokens</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion_tokens</code>는 바로 앞 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 값과 같습니다.</p>
<p style="margin:6px 0;">앞 셀을 다시 실행했다면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests</code>가 실행한 만큼 늘어 있습니다. free 요금제라 60초 안에 넷째 상담 요청부터는 429입니다.</p>
</div>

### 6-3. 키 폐기와 401 확인

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 5-2를 참고합니다.
세부구현: 폐기는 httpx.delete로 보냅니다. 폐기 주소는 SERVING_URL 뒤에 6-1에서 키를 발급할 때 쓴 관리자 주소와 6-1 발급 응답의 key_id를 붙인 f-문자열입니다. 폐기는 관리자 주소라 관리자 키 헤더를 보내고, timeout은 REQUEST_TIMEOUT입니다.
```

</details>

In [ ]:
# 🖐️ 함께 따라하기: 관리자 키 헤더를 붙여 6-1에서 발급한 키의 폐기 주소에 DELETE 요청을 보내고, 응답을 lease_revoke_response 에 담습니다
lease_revoke_response = ...

# 폐기한 키로 전월세 질문을 한 번 더 보냅니다
lease_after_response = httpx.post(f"{SERVING_URL}/v1/lease/chat", headers=lease_headers, json={"question": lease_questions[1]["question"]}, timeout=REQUEST_TIMEOUT)

# 두 단계의 상태 코드와 본문을 한 테이블로 모읍니다
lease_revoke_table = pd.DataFrame([
    {"단계": "폐기", "상태 코드": lease_revoke_response.status_code, "본문": lease_revoke_response.json()},
    {"단계": "폐기한 키로 상담", "상태 코드": lease_after_response.status_code, "본문": lease_after_response.json()},
])
display(lease_revoke_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 폐기 행은 200과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;revoked&quot;: True</code>, 폐기한 키로 상담 행은 401과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">invalid_service_key</code>입니다.</p>
<p style="margin:6px 0;">폐기 행이 404이면 주소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>가 6-1의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>와 다르거나, 이 셀을 두 번 실행해 이미 폐기한 키입니다. 6-1부터 다시 실행합니다.</p>
</div>

## 7. 핵심 정리

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SQLite 키 저장소:</strong> 서비스 키와 사용량을 파일 하나(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/service.db</code>)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_keys</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 테이블에 저장합니다. 서버가 요청마다 DB를 읽으므로 발급과 폐기가 서버를 다시 켜지 않아도 바로 반영됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">해시 저장:</strong> 키 원문 대신 SHA-256 해시(64자)를 저장하고, 요청의 키를 해시로 바꿔 찾습니다. DB가 새어도 원문을 알 수 없고, 원문은 발급 응답에서 한 번만 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">관리자 주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/admin/keys</code> 묶음은 라우터를 만들 때 붙인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_admin</code> 의존성으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Admin-Key</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compare_digest</code>로 비교합니다. 관리자 키가 없거나 틀리면 403 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">admin_only</code>이고, 발급은 201로 답합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">사용량 기록:</strong> 토큰 수는 답이 나온 뒤에야 알 수 있어 엔드포인트 안에서 기록합니다. 일반 주소는 답을 받은 뒤, 스트리밍 주소는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 이벤트를 만났을 때 입력, 캐시 적중, 출력 토큰 수를 DB에 씁니다. DB 함수는 lifespan이 연 aiosqlite 연결 하나를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_db</code> 의존성으로 받아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>하므로, 쓰는 동안에도 다른 요청을 받습니다. 경로 매개변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">desk</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Literal</code>)로 노무와 전월세 창구를 한 엔드포인트로 처리합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">사용량 조회:</strong> 서비스 키 주인은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /v1/usage</code>로 자기 키의 합계를, 관리자는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /v1/admin/keys/{key_id}/usage</code>로 아무 키의 합계를 봅니다. 두 주소는 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage_summary</code>로 계산해 값이 같고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">start</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">end</code>로 기간을 정할 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">폐기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DELETE /v1/admin/keys/{key_id}</code>는 행을 지우지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">revoked_at</code>을 채웁니다. 폐기한 키는 다음 요청부터 401이고, 다시 폐기하면 404입니다. 사용량 기록은 관리자가 계속 조회할 수 있습니다.</li>
</ul>
<p style="margin:6px 0;">다만 SQLite 파일은 서버 한 대의 디스크에 있습니다. 같은 앱을 서버 두 대로 띄워 각자 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">service.db</code>를 가지면, 1번 서버에서 발급한 키를 2번 서버는 몰라 401로 거절하고, 사용량도 서버마다 따로 쌓입니다. 서버 여러 대로 띄울 때는 모든 서버가 함께 읽고 쓰는 공유 DB를 써야 합니다. 모델 배포 단원에서는 FastAPI를 GCP VM 한 대에 띄우고, 이 파일을 그 VM의 디스크에 남겨 둡니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">부록 01(선택)을 이어서 하면</strong> 터미널 1의 vLLM 서버는 그대로 둡니다. 부록 01은 터미널 2에 부록 폴더의 앱을, 새 터미널 3에 LiteLLM Proxy를 띄웁니다. 오늘 실습을 여기서 마칠 때만 아래 순서로 끕니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">오늘 실습을 여기서 마칠 때</summary>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>(FastAPI 서버)와 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> vLLM 서버)에서 차례로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> vLLM 서버가 꺼진 것입니다.</p>
<p style="margin:6px 0;">오늘 실습을 마치면 RunPod 콘솔에서 파드를 종료(Terminate)합니다. 정지(Stop)해도 다음에 같은 GPU로 다시 켜지지 않는 경우가 많습니다. 남길 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output</code> 등)이 있으면 종료 전에 내 PC로 받아 둡니다.</p>
</div>

</details>